# 饮品数据标准化与映射流程

本 Notebook 用于将外部采集数据标准化并映射到系统已有商品池（`products` / `orders` / `order_items`），并对未匹配项执行降权兜底。

## 0. 参数配置

- `EXTERNAL_CSV_PATH`: 外部采集 CSV 文件路径
- `PRODUCT_CATALOG_PATH`: 现有商品池
- `FALLBACK_RATIO`: 兜底比例

In [1]:
import re
import random
import pandas as pd
from pathlib import Path

random.seed(2026)

EXTERNAL_CSV_PATH = Path("./奶茶店每日订单.csv")
PRODUCT_CATALOG_PATH = Path("./product_catalog.csv")
OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FALLBACK_RATIO = 0.08

print("external:", EXTERNAL_CSV_PATH)
print("catalog :", PRODUCT_CATALOG_PATH)
print("out dir :", OUT_DIR.resolve())

external: 奶茶店每日订单.csv
catalog : product_catalog.csv
out dir : C:\Users\user\Downloads\project\backend\jupyter\outputs


## 1. 读取采集数据与商品池

In [3]:
ext_df = pd.read_csv(EXTERNAL_CSV_PATH)

def _is_garbled_name_series(name_series: pd.Series) -> bool:
    sample = name_series.dropna().astype(str).head(10).tolist()
    if not sample:
        return True
    # 只要大部分是问号占位，就判定为乱码
    bad = 0
    for s in sample:
        s_strip = s.strip()
        if not s_strip:
            bad += 1
            continue
        if set(s_strip) == {"?"}:
            bad += 1
    return bad >= max(1, int(len(sample) * 0.6))


def _rebuild_catalog_from_seed(seed_path: Path, out_path: Path) -> pd.DataFrame:
    txt = seed_path.read_text(encoding="utf-8")
    pattern = r'\(cat_map\["([^"]+)"\],\s*"([^"]+)",\s*Decimal\("([\d.]+)"\),\s*"([^"]+)"\)'
    rows = []
    for i, m in enumerate(re.finditer(pattern, txt), start=1):
        category_name, name, base_price, image_url = m.groups()
        rows.append({
            "id": i,
            "category_name": category_name,
            "name": name,
            "base_price": float(base_price),
            "image_url": image_url,
            "is_active": 1,
        })
    if not rows:
        raise ValueError("从 seed_data.py 中未解析到商品数据")
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    return df

catalog_df = None
for enc in ["utf-8", "utf-8-sig", "gbk", "gb18030", "latin1"]:
    try:
        tmp_df = pd.read_csv(PRODUCT_CATALOG_PATH, encoding=enc)
        tmp_df = tmp_df.loc[:, ~tmp_df.columns.str.startswith("Unnamed")]
        if "name" not in tmp_df.columns:
            continue
        if _is_garbled_name_series(tmp_df["name"]):
            print(f"catalog encoding {enc} loaded but appears garbled, will rebuild from seed_data.py")
            continue
        catalog_df = tmp_df
        print(f"catalog encoding: {enc}")
        break
    except Exception:
        continue

if catalog_df is None:
    seed_path = Path("../seed_data.py")
    rebuilt_path = Path("./product_catalog_clean.csv")
    catalog_df = _rebuild_catalog_from_seed(seed_path, rebuilt_path)
    PRODUCT_CATALOG_PATH = rebuilt_path
    print(f"catalog rebuilt from {seed_path} -> {rebuilt_path}")

print("external rows:", len(ext_df))
print("external cols:", list(ext_df.columns))
display(ext_df.head(5))
print("catalog rows :", len(catalog_df))
print("catalog cols :", list(catalog_df.columns))
display(catalog_df.head(5))

catalog encoding utf-8 loaded but appears garbled, will rebuild from seed_data.py
catalog encoding utf-8-sig loaded but appears garbled, will rebuild from seed_data.py
catalog encoding gbk loaded but appears garbled, will rebuild from seed_data.py
catalog encoding gb18030 loaded but appears garbled, will rebuild from seed_data.py
catalog encoding latin1 loaded but appears garbled, will rebuild from seed_data.py
catalog rebuilt from ..\seed_data.py -> product_catalog_clean.csv
external rows: 3000
external cols: ['日期', '星期', '商品', '天气', '是否周末', '是否促销', '销量', '客单价', '销售额']


,日期,星期,商品,天气,是否周末,是否促销,销量,客单价,销售额
0,5/21/2024,周二,芋泥波波,雨,False,False,12,11,132
1,5/9/2024,周四,芋泥波波,晴,False,True,39,18,702
2,5/11/2024,周六,柠檬茶,雪,True,False,52,19,988
3,3/1/2024,周五,杨枝甘露,雨,False,False,26,20,520
4,4/18/2024,周四,芋泥波波,雨,False,False,21,17,357


catalog rows : 57
catalog cols : ['id', 'category_name', 'name', 'base_price', 'image_url', 'is_active']


,id,category_name,name,base_price,image_url,is_active
0,1,热销,黑糖珍珠奶茶,10.0,/assets/rm01.jpg,1
1,2,热销,芝士奶盖抹茶,14.0,/assets/rm02.jpg,1
2,3,热销,焦糖布丁奶茶,13.0,/assets/rm03.jpg,1
3,4,招牌人气,金凤乌龙奶茶,12.0,/assets/zp01.jpg,1
4,5,招牌人气,多多绿茶奶茶,12.0,/assets/zp02.jpg,1


## 2. 字段标准化（外部字段 -> 中间标准字段）

根据采集 CSV 的实际中文列名进行映射：

- `日期` -> `order_date`
- `商品` -> `ext_product_name`
- `客单价` -> `ext_price`
- `销量` -> `ext_qty`

其余列（`星期`、`天气`、`是否周末`、`是否促销`、`销售额`）保留为扩展特征。

In [10]:
rename_map = {
    "日期": "order_date",
    "商品": "ext_product_name",
    "客单价": "ext_price",
    "销量": "ext_qty",
}

std_df = ext_df.rename(columns=rename_map).copy()
std_df["order_date"] = pd.to_datetime(std_df["order_date"], errors="coerce")
std_df["ext_price"] = pd.to_numeric(std_df["ext_price"], errors="coerce")
std_df["ext_qty"] = pd.to_numeric(std_df["ext_qty"], errors="coerce").fillna(1).astype(int)

print("\n=== 标准化后前5行 ===")
display(std_df.head(5))
print("=== 标准化后缺失值检查 ===")
print(std_df[["order_date", "ext_product_name", "ext_price", "ext_qty"]].isnull().sum())
print("\n=== 外部商品唯一值 ===")
print(std_df["ext_product_name"].unique())


=== 标准化后前5行 ===


,order_date,星期,ext_product_name,天气,是否周末,是否促销,ext_qty,ext_price,销售额
0,2024-05-21,周二,芋泥波波,雨,False,False,12,11,132
1,2024-05-09,周四,芋泥波波,晴,False,True,39,18,702
2,2024-05-11,周六,柠檬茶,雪,True,False,52,19,988
3,2024-03-01,周五,杨枝甘露,雨,False,False,26,20,520
4,2024-04-18,周四,芋泥波波,雨,False,False,21,17,357


=== 标准化后缺失值检查 ===
order_date          0
ext_product_name    0
ext_price           0
ext_qty             0
dtype: int64

=== 外部商品唯一值 ===
['芋泥波波' '柠檬茶' '杨枝甘露' '珍珠奶茶' '芝士奶盖' '可可奶茶' '水果茶' '冰美式' '抹茶拿铁' '青稞奶茶']


## 3. 商品名归一化与映射

映射优先级：
1. 完全匹配（exact）
2. 关键词匹配（keyword）
3. 兜底（fallback）

In [5]:
def normalize_name(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", "", x)
    x = re.sub(r"(大杯|中杯|小杯|热饮|冷饮|去冰|少冰|无糖|少糖|半糖|加冰)$", "", x)
    return x

catalog_df["name_norm"] = catalog_df["name"].map(normalize_name)
std_df["ext_name_norm"] = std_df["ext_product_name"].map(normalize_name)

exact_map = dict(zip(catalog_df["name_norm"], catalog_df["name"]))

ext_unique = sorted(std_df["ext_name_norm"].unique())
catalog_names = sorted(catalog_df["name"].tolist())
print("=== 外部唯一商品 ===")
print(ext_unique)
print("\n=== 系统商品池 ===")
print(catalog_names)

keyword_rules = {
    "珍珠奶茶": "黑糖珍珠奶茶",
    "珍珠": "黑糖珍珠奶茶",
    "芋泥": "芋泥波波奶茶",
    "波波": "芋泥波波奶茶",
    "芝士奶盖": "芝士奶盖抹茶",
    "芝士": "芝士奶盖抹茶",
    "抹茶拿铁": "芝士奶盖抹茶",
    "抹茶": "芝士奶盖抹茶",
    "杨枝甘露": "杨枝甘露奶茶",
    "柠檬茶": "百香果绿茶",
    "柠檬": "百香果绿茶",
    "水果茶": "百香果绿茶",
    "可可奶茶": "红丝绒可可奶茶",
    "可可": "红丝绒可可奶茶",
    "冰美式": "金凤乌龙奶茶",
    "美式": "金凤乌龙奶茶",
    "拿铁": "焦糖布丁奶茶",
}

fallback_candidates = catalog_df["name"].tolist()

mapped_name = []
match_type = []

for n in std_df["ext_name_norm"]:
    if n in exact_map:
        mapped_name.append(exact_map[n])
        match_type.append("exact")
        continue

    hit = None
    for k, v in keyword_rules.items():
        if k in n:
            hit = v
            break

    if hit:
        mapped_name.append(hit)
        match_type.append("keyword")
    else:
        mapped_name.append(random.choice(fallback_candidates))
        match_type.append("fallback")

std_df["mapped_product_name"] = mapped_name
std_df["match_type"] = match_type

print("\n=== 映射结果前20行 ===")
display(std_df[["ext_product_name", "mapped_product_name", "match_type"]].head(20))
print("\n=== 匹配类型分布 ===")
print(std_df["match_type"].value_counts())
print(std_df["match_type"].value_counts(normalize=True).round(4))

=== 外部唯一商品 ===
['冰美式', '可可奶茶', '抹茶拿铁', '杨枝甘露', '柠檬茶', '水果茶', '珍珠奶茶', '芋泥波波', '芝士奶盖', '青稞奶茶']

=== 系统商品池 ===
['低脂抹茶拿铁', '冰冻葡萄奶茶', '冰镇哈密瓜奶茶', '双拼渐变奶茶', '多多绿茶奶茶', '奇异果百香奶茶', '奥利奥脆脆奶茶', '小蛋糕（抹茶红豆）', '小蛋糕（樱花龙眼）', '小蛋糕（芒果百香果）', '小蛋糕（草莓巧克力）', '小蛋糕（蓝莓酸奶）', '无糖生椰抹茶', '曲奇', '杨枝甘露奶茶', '柚子蜂蜜奶茶', '梦幻彩虹奶茶', '椰果青柠奶茶', '椰椰生椰拿铁', '樱花芝士奶盖茶', '樱花草莓奶茶', '橙香茉莉奶茶', '火龙果椰香奶茶', '炸鸡块', '焦糖布丁奶茶', '燕麦奶茶', '燕麦红枣奶茶', '玫瑰荔枝奶茶', '甜甜圈（巧克力果仁）', '甜甜圈（椰子椰奶）', '甜甜圈（草莓巧克力）', '甜甜圈（草莓椰奶）', '甜甜圈（草莓椰子）', '甜甜圈（草莓樱花）', '百香果绿茶', '紫薯珍珠奶茶', '红丝绒可可奶茶', '芒果爆柠奶茶', '芝士奶盖抹茶', '芝士蛋糕', '草莓芝士奶盖', '菠萝椰香奶茶', '蓝柑冰奶茶', '蓝莓优格奶茶', '薄荷青柠奶茶', '薯条', '薰衣草奶茶', '蜂蜜柠檬奶茶', '蜂蜜柠檬普洱奶茶', '西瓜冰奶茶', '豆乳燕麦奶茶', '豆乳黑芝麻奶茶', '金凤乌龙奶茶', '鸡米花', '麻薯球', '黑糖珍珠奶茶', '黑豆芝麻豆浆奶茶']

=== 映射结果前20行 ===


,ext_product_name,mapped_product_name,match_type
0,芋泥波波,芋泥波波奶茶,keyword
1,芋泥波波,芋泥波波奶茶,keyword
2,柠檬茶,百香果绿茶,keyword
3,杨枝甘露,杨枝甘露奶茶,keyword
4,芋泥波波,芋泥波波奶茶,keyword
5,芋泥波波,芋泥波波奶茶,keyword
6,珍珠奶茶,黑糖珍珠奶茶,keyword
7,芝士奶盖,芝士奶盖抹茶,keyword
8,芋泥波波,芋泥波波奶茶,keyword
9,可可奶茶,红丝绒可可奶茶,keyword



=== 匹配类型分布 ===
match_type
keyword     2704
fallback     296
Name: count, dtype: int64
match_type
keyword     0.9013
fallback    0.0987
Name: proportion, dtype: float64


## 4. 兜底降权

仅保留部分 `fallback`，防止其影响热门规则分布。

In [6]:
is_fallback = std_df["match_type"] == "fallback"
fallback_df = std_df[is_fallback].sample(frac=FALLBACK_RATIO, random_state=2026)
main_df = std_df[~is_fallback]
final_df = pd.concat([main_df, fallback_df], ignore_index=True)

print("before:", len(std_df), "after:", len(final_df))
print(final_df["match_type"].value_counts(normalize=True).round(4))

before: 3000 after: 2728
match_type
keyword     0.9912
fallback    0.0088
Name: proportion, dtype: float64


## 5. 生成可写入业务表的结果（示例）

In [7]:
out_df = final_df.copy()
out_df["qty"] = out_df["ext_qty"].clip(lower=1)
out_df["unit_price"] = out_df["ext_price"].fillna(out_df["ext_price"].median())
out_df["line_total"] = out_df["qty"] * out_df["unit_price"]
out_df["product_name_snap"] = out_df["mapped_product_name"]

mapping_audit = out_df[["ext_product_name", "mapped_product_name", "match_type"]].drop_duplicates()

out_df.to_csv(OUT_DIR / "standardized_order_items_seed.csv", index=False)
mapping_audit.to_csv(OUT_DIR / "product_mapping_audit.csv", index=False)

display(out_df.head(10))
print("saved:", OUT_DIR / "standardized_order_items_seed.csv")
print("saved:", OUT_DIR / "product_mapping_audit.csv")

,order_date,星期,ext_product_name,天气,是否周末,是否促销,ext_qty,ext_price,销售额,ext_name_norm,mapped_product_name,match_type,qty,unit_price,line_total,product_name_snap
0,2024-05-21,周二,芋泥波波,雨,False,False,12,11,132,芋泥波波,芋泥波波奶茶,keyword,12,11,132,芋泥波波奶茶
1,2024-05-09,周四,芋泥波波,晴,False,True,39,18,702,芋泥波波,芋泥波波奶茶,keyword,39,18,702,芋泥波波奶茶
2,2024-05-11,周六,柠檬茶,雪,True,False,52,19,988,柠檬茶,百香果绿茶,keyword,52,19,988,百香果绿茶
3,2024-03-01,周五,杨枝甘露,雨,False,False,26,20,520,杨枝甘露,杨枝甘露奶茶,keyword,26,20,520,杨枝甘露奶茶
4,2024-04-18,周四,芋泥波波,雨,False,False,21,17,357,芋泥波波,芋泥波波奶茶,keyword,21,17,357,芋泥波波奶茶
5,2024-04-18,周四,芋泥波波,雪,False,False,24,19,456,芋泥波波,芋泥波波奶茶,keyword,24,19,456,芋泥波波奶茶
6,2024-03-09,周六,珍珠奶茶,雨,True,False,35,12,420,珍珠奶茶,黑糖珍珠奶茶,keyword,35,12,420,黑糖珍珠奶茶
7,2024-04-28,周日,芝士奶盖,雨,True,False,40,14,560,芝士奶盖,芝士奶盖抹茶,keyword,40,14,560,芝士奶盖抹茶
8,2024-05-27,周一,芋泥波波,雪,False,False,17,17,289,芋泥波波,芋泥波波奶茶,keyword,17,17,289,芋泥波波奶茶
9,2024-05-21,周二,可可奶茶,雨,False,False,17,13,221,可可奶茶,红丝绒可可奶茶,keyword,17,13,221,红丝绒可可奶茶


saved: outputs\standardized_order_items_seed.csv
saved: outputs\product_mapping_audit.csv


## 6. 去重前后对比

In [8]:
before_rows = len(ext_df)
dup_cnt = ext_df.duplicated().sum()
ext_df = ext_df.drop_duplicates()
after_rows = len(ext_df)

print("去重前记录数:", before_rows)
print("重复记录数:", dup_cnt)
print("去重后记录数:", after_rows)

去重前记录数: 3000
重复记录数: 4
去重后记录数: 2996


## 打印结果

In [9]:
s = pd.read_csv('./outputs/standardized_order_items_seed.csv')
p = pd.read_csv('./outputs/product_mapping_audit.csv')

display(s.head(10))
display(p.head(10))

,order_date,星期,ext_product_name,天气,是否周末,是否促销,ext_qty,ext_price,销售额,ext_name_norm,mapped_product_name,match_type,qty,unit_price,line_total,product_name_snap
0,2024-05-21,周二,芋泥波波,雨,False,False,12,11,132,芋泥波波,芋泥波波奶茶,keyword,12,11,132,芋泥波波奶茶
1,2024-05-09,周四,芋泥波波,晴,False,True,39,18,702,芋泥波波,芋泥波波奶茶,keyword,39,18,702,芋泥波波奶茶
2,2024-05-11,周六,柠檬茶,雪,True,False,52,19,988,柠檬茶,百香果绿茶,keyword,52,19,988,百香果绿茶
3,2024-03-01,周五,杨枝甘露,雨,False,False,26,20,520,杨枝甘露,杨枝甘露奶茶,keyword,26,20,520,杨枝甘露奶茶
4,2024-04-18,周四,芋泥波波,雨,False,False,21,17,357,芋泥波波,芋泥波波奶茶,keyword,21,17,357,芋泥波波奶茶
5,2024-04-18,周四,芋泥波波,雪,False,False,24,19,456,芋泥波波,芋泥波波奶茶,keyword,24,19,456,芋泥波波奶茶
6,2024-03-09,周六,珍珠奶茶,雨,True,False,35,12,420,珍珠奶茶,黑糖珍珠奶茶,keyword,35,12,420,黑糖珍珠奶茶
7,2024-04-28,周日,芝士奶盖,雨,True,False,40,14,560,芝士奶盖,芝士奶盖抹茶,keyword,40,14,560,芝士奶盖抹茶
8,2024-05-27,周一,芋泥波波,雪,False,False,17,17,289,芋泥波波,芋泥波波奶茶,keyword,17,17,289,芋泥波波奶茶
9,2024-05-21,周二,可可奶茶,雨,False,False,17,13,221,可可奶茶,红丝绒可可奶茶,keyword,17,13,221,红丝绒可可奶茶


,ext_product_name,mapped_product_name,match_type
0,芋泥波波,芋泥波波奶茶,keyword
1,柠檬茶,百香果绿茶,keyword
2,杨枝甘露,杨枝甘露奶茶,keyword
3,珍珠奶茶,黑糖珍珠奶茶,keyword
4,芝士奶盖,芝士奶盖抹茶,keyword
5,可可奶茶,红丝绒可可奶茶,keyword
6,水果茶,百香果绿茶,keyword
7,冰美式,金凤乌龙奶茶,keyword
8,抹茶拿铁,芝士奶盖抹茶,keyword
9,青稞奶茶,紫薯珍珠奶茶,fallback
